# Data Scraping

To forecast DK1 electricity prices, we first need data that describes the price itself and the conditions that influence it. We collect day-ahead prices, observed weather, previous-run weather forecasts, and consumption data. The raw files are saved in `data/` and then processed before they are used by the model.

| Data | Why it is needed | Saved file |
|------|------------------|------------|
| Day-ahead prices | Target variable and historical price information | `data/day_ahead_prices_dk1_raw.csv` |
| Weather actuals | Historical weather base | `data/weather_actuals_raw.csv` |
| Weather forecasts | Used to estimate realistic forecast errors | `data/weather_forecasts_raw.csv` |
| Consumption | Demand context used later in the project | `data/consumption_dk1_raw.csv` |

All raw datasets can be collected with one call:

In [ ]:
from src.data.data_collection import fetch_all

results = fetch_all(start="2021-01-01", end="2026-04-28", price_area="DK1")

As a quick check, we plot prices and actual weather for the same period. This helps confirm that the scraped time series line up and that the weather variables are relevant for price forecasting.

In [ ]:
from src.analysis.forecast_analysis import plot_raw_price_and_weather

fig = plot_raw_price_and_weather(start="2025-01-01", end="2025-01-14")

Before the data can be used for prediction, it must be changed into the same format the model will see in practice. The key issue is weather: at forecast time, we do not know future actual weather. We only know weather forecasts.

Real previous-run weather forecasts are only available from 2025 onwards. We therefore use that period to estimate typical forecast errors, and then use those errors to generate forecast-like weather inputs for the full historical period.

The data processing script estimates weather forecast errors, creates forecast-like weather data, joins the electricity price target, and adds price lags that would have been known at the forecast issue time.

In [ ]:
!python -m src.data.data_processing

The first plot shows how weather forecast error changes with the forecast horizon. The wider distributions at longer horizons show why the simulated forecasts become more uncertain further into the future.

In [ ]:
from src.data.data_processing import plot_weather_error_distributions

plot_weather_error_distributions()

The next plot shows the final weather structure used by the model: historical actual weather before the issue time, followed by the generated 5-day forecast window. The actual weather in the forecast window is shown only for comparison.

In [ ]:
from src.data.data_processing import plot_weather_forecast_dashboard_style

fig_weather = plot_weather_forecast_dashboard_style(
    issue_time="2026-04-23 00:00",
    ctx_days=7,
    show_actuals=True,
)

The final model input is `data/model_dataset.parquet`. It contains one row per forecast horizon, with forecast-like weather variables, time features, the matching price target, and price lags based only on information available at `issue_time`.

In [ ]:
import pandas as pd

model_data = pd.read_parquet("data/model_dataset.parquet")
model_data.head()